In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.graphics.tsaplots import acf, plot_acf, plot_pacf

# Aufgabe 1: CO<sub>2</sub> Monitoring

In [ ]:
def plot_data(df: pd.DataFrame, lin: pd.Series = None) -> None:
    plt.figure(figsize=(20, 8))

    if lin is not None:
        plt.plot(df.index.year, df["value"], color="red")
        plt.plot(df.index.year, lin, color="orange", label="Linearer Trend")
        plt.ylabel(r"Log(CO$_2$(t) / CO$_2$(1969))")
        plt.legend()
    else:
        plt.plot(df["value"], color="red")
        plt.ylabel(r"CO$_2$")

    plt.title(r"Mauna Loa: CO$_2$-Werte")
    plt.xlabel("Datum")
    plt.grid(alpha=0.5)
    plt.margins(x=0)
    plt.show()

In [ ]:
df = pd.read_csv("co2_mlo_surface-flask_1_ccgg_event.txt", sep=r"\s+", comment="#", header=0)
# display(df.dtypes)
# display(df)
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.set_index("datetime")[["value"]].sort_index()

plot_data(df)

#### Behandlung von Lücken / fehlenden Werten

In [ ]:
# Werte wie -999 und 1000 sind offensichtlich Messfehler, daher filtern
df[(df["value"] < -990) | (df["value"] > 990)] = None
# Mittelwert bei doppeltem Datum
df = df.groupby(level=0).mean()

plot_data(df)

In [ ]:
# Ausgabe der Lücken
series = df.index.to_series()
series = series.groupby(level=0).mean()

m = series.diff() > pd.Timedelta(days=1)

gaps = pd.DataFrame(
    {
        "start": series[m].shift(1),
        "end": series[m],
    }
)
gaps["length"] = gaps["end"] - gaps["start"]
gaps = gaps.reset_index(drop=True)

print("Lücken der Zeitreihe:")
display(gaps.sort_values(by="length", ascending=False).head(15))

# Für Quotienten Test
year_1969 = df.loc[: pd.to_datetime("1969-12-31 23:59:59+00:00")].copy().resample("YE").mean()

# Zuschneiden der Zeitreihe, übrige Lücken < 30 Tage
df = df.loc[pd.to_datetime("1977-01-01 20:00:00+00:00") :].copy()

# Monatliches Mittel zum Füllen der Lücken
df = df.resample("MS").mean()

plot_data(df)

#### Jährliches Mittel & Wachstumsrate

In [ ]:
yearly = df.resample("YE").mean()

plot_data(yearly)

print("Global:")
display(yearly.describe().transpose())
print("Pro Jahrzehnt:")
display(yearly.groupby((yearly.index.year // 10) * 10).describe())

rate = {"1980-1989": 0, "2000-2009": 0, "2010-2019": 0, "2020-2024": 0}

for index, year in enumerate([(1980, 1989), (2000, 2009), (2010, 2019), (2020, 2024)]):
    start = yearly[yearly.index.year == year[0]].iloc[0, 0]
    end = yearly[yearly.index.year == year[1]].iloc[0, 0]
    # Berechnung jährlicher Wachstumsrate
    rate[list(rate.keys())[index]] = round(float((end / start) ** (1 / (year[1] - year[0])) - 1) * 100, 4)

print("Jährliche Wachstumsrate in %:")
string = ""
for key, val in rate.items():
    string = string + f"\t{key}: {val}\n"
print(string)

<font color="orange">
Die jährliche Wachstumsrate steigt kontinuierlich an. Die Zunahme scheint dabei nicht linear zu verlaufen, was auf eine beschleunigte Entwicklung der CO<sub>2</sub>-Werte hindeutet.
</font>

In [ ]:
# Quotienten Test
log = np.log(yearly / year_1969.iloc[0, 0])

# Linearer Trend
z = np.polyfit(yearly.index.year, log["value"], 1)
p = np.poly1d(z)

plot_data(log, p(yearly.index.year))

<font color="orange">
Der Log-Plot verläuft sehr nahe des linearen Trends (mit leichten Abweichungen). Das deutet auf ein annähernd exponentielles Wachstum der CO<sub>2</sub>-Werte hin.
</font>

# Aufgabe 2: Zeitreihen Eigenschaften und Vorhersage

In [ ]:
df = pd.read_csv("FreiburgNiederschlag.csv")
df["category"] = pd.to_datetime(df["category"], format="mixed")
df = df.rename(columns={"category": "date", "Niederschlag": "value"})
df = df.set_index("date").sort_index()

In [ ]:
# Trend
x = mdates.date2num(df.index)
p = np.poly1d(np.polyfit(x, df["value"], 1))
df["trend"] = p(x)

df["detrended"] = df["value"] - df["trend"]

plt.figure(figsize=(20, 8))

plt.plot(df.index, df["value"], color="lightblue", alpha=0.3, label="Niederschlag")
plt.plot(df.index, df["detrended"], color="blue", alpha=0.8, label="Niederschlag (Trendbereinigt)")
plt.plot(df.index, df["trend"], color="blue", alpha=1, label="Trend", linestyle="--")

plt.title("Freiburg Messdaten (Niederschlag) - Trend")
plt.xlabel("Datum")
plt.ylabel("Niederschlag in mm")
plt.grid(axis="y", alpha=0.5)
plt.margins(x=0)
plt.legend()
plt.show()

In [ ]:
# Jährliche Saisonalität
df["month"] = df.index.month
df["year"] = df.index.year

seasonal = df.groupby("month")["detrended"].mean()

plt.figure(figsize=(20, 8))

for i, y in enumerate(df["year"].unique()):
    data = df[df["year"] == y]
    if i == 0:
        plt.plot(data["month"], data["detrended"], color="gray", alpha=0.15, label="alle Jahre")
    else:
        plt.plot(data["month"], data["detrended"], color="gray", alpha=0.15)
plt.plot(seasonal, color="blue", alpha=1, label="langjähriges Monatsmittel", linewidth=3)

plt.title("Freiburg Messdaten (Niederschlag) - Saisonalität")
plt.xlabel("Monat")
plt.ylabel("Niederschlag in mm")
plt.xticks(
    ticks=range(1, 13), labels=pd.Series(range(1, 13)).map(lambda x: pd.to_datetime(x, format="%m").strftime("%b"))
)
plt.grid(axis="both", alpha=0.5)
plt.margins(x=0)
plt.legend()
plt.show()

df = df.join(seasonal, on="month", rsuffix="_seasonal")

In [ ]:
# Bereinigte Zeitreihe
df["resid"] = df["detrended"] - df["detrended_seasonal"]

plt.figure(figsize=(20, 8))

plt.plot(df.index, df["value"], color="lightblue", alpha=0.3, label="Niederschlag")
plt.plot(df.index, df["resid"], color="blue", alpha=0.9, label="Niederschlag (Residuum)")

plt.title("Freiburg Messdaten (Niederschlag) - Bereinigt")
plt.xlabel("Datum")
plt.ylabel("Niederschlag in mm")
plt.grid(axis="both", alpha=0.5)
plt.margins(x=0)
plt.legend()
plt.show()

In [ ]:
# Autokorrelation
vals = acf(df["resid"].dropna(), nlags=12)
lags = [1, 3, 6, 9, 12]

plt.stem(lags, [vals[lag] for lag in lags])
plt.xlabel("Lag")
plt.ylabel("Autokorrelation")
plt.grid(which="both", alpha=0.5)
plt.show()

<font color="orange">
TODO Interpretation
</font>

In [ ]:
# TODO